# Async in python

## Coroutine objects

In [1]:
import asyncio

In [ ]:
async def fetch_data():
    print ("started")

In [ ]:
fetch_data()

<coroutine object fetch_data at 0x000001C57BD79780>

In [ ]:
coro = fetch_data()

In [ ]:
await coro

started


In [ ]:
async def fetch_data():
    print("A")
    print("B")

In [ ]:
coro = fetch_data() # wont be executed because creating the coroutine object does not start it.

print("C")

C


In [ ]:
async def fetch_data():
    print("A")
    print("B")

coro = fetch_data() # wont be executed yet

print("C") # 1st

await coro # 2nd

print("D") # 3rd

C
A
B
D


C:\Users\hamza\AppData\Local\Temp\ipykernel_2508\4187357328.py:5: RuntimeWarning: coroutine 'fetch_data' was never awaited
  coro = fetch_data() # wont be executed yet


In [ ]:
async def some_io(name, delay):
    print(f"{name}: starting I/O")
    await asyncio.sleep(delay)
    print(f"{name}: I/O completed")

async def A ():
    print ("A1")
    await some_io("A", 3)
    print("A2")

async def B ():
    print ("B1")
    await some_io("B", 1)
    print("B2")
# Note we can have both A and B in one thread

In [ ]:
coro_A  = A() # coroutine objects means "run A and wait until its finished"
coro_B  = B() # after A finished "now run B"

In [ ]:
result_A  = await coro_A # The value returned by the coroutine
result_B  = await coro_B 

A1
A: starting I/O
A: I/O completed
A2
B1
B: starting I/O
B: I/O completed
B2


``` text
await A()
   │
   ├── A1
   ├── A starts I/O
   ├── A waits 3 seconds
   ├── A I/O completes
   └── A2
          │
          ↓
     await B()
          │
          ├── B1
          ├── B starts I/O
          ├── B waits 1 second
          ├── B I/O completes
          └── B2
```

## Scheduling 

In [ ]:
# task_A, task_b are awaitable 
task_A = asyncio.create_task(A()) # Means schedule A, it does not create a thread. It schedules the coroutine as a Task on the event loop.
task_B = asyncio.create_task(B()) # Schedule B as another Task on the event loop.

A1
A: starting I/O
B1
B: starting I/O


B: I/O completed
B2
A: I/O completed
A2


In [ ]:
type(task_A)

_asyncio.Task

``` text
Event Loop
│
├── Task A → running/waiting
└── Task B → running/waiting
```
``` text
A1
A: starting I/O
      ↓
A waits 3s
      ↓
B1
B: starting I/O
      ↓
B waits 1s
      ↓
B's I/O completes
      ↓
B2
      ↓
A's I/O completes
      ↓
A2
```

## Event loop

In [ ]:
async def main():
    print("Hello")

In [ ]:
main() # create a coroutine object, nothing drives it.

<coroutine object main at 0x000001C57BE67340>

In [ ]:
# asyncio.run(main()) # wont run in Jupiter as it has already running loop, but it is essential for starting event loop in .py

In [ ]:
async def A():
    print("A1")
    await asyncio.sleep(2)
    print("A2")
async def main():
    task = asyncio.create_task(A())

In [ ]:
await main() # In .py file: await main()

A1


A2


In [ ]:
async def A():
    print ("A1")
    await asyncio.sleep(3)
    print ("A2")

async def main():
    task = asyncio.create_task(A())

    print ("B")

    await task # here `main` says: I need task A to finish before I can continue 

    print ("B")

In [ ]:
await main()

B
A1
A2
B


## asyncio.gather()

In [ ]:
async def A():
    print ("A")

async def B():
    print ("B")

async def C():
    print ("C")

In [ ]:
task_A = asyncio.create_task(A())
task_B = asyncio.create_task(B())


A
B


In [ ]:
await task_A
await task_B

In [ ]:
results = await asyncio.gather( # means "Run these awaitables concurrently and give me their results when they have all completed"
    A(), 
    B(), 
    C() 
)

A
B
C


In [ ]:
async def A():
    await asyncio.sleep(3)
    print("A")
    return "A"

async def B():
    await asyncio.sleep(1)
    print("B")
    return "B"

async def C():
    await asyncio.sleep(2)
    print("C")
    return "C"

In [ ]:
results = await asyncio.gather(  
    A(), 
    B(), 
    C() 
)


B
C
A


In [ ]:
print(results)

['A', 'B', 'C']


## cancel()

In [ ]:
async def work():
    print ("1: started") # 1

    try:
        await asyncio.sleep(10)
        print("2: finished sleeping")

    except asyncio.CancelledError:
        print("3: cancellation error") # 3

async def main():
    task = asyncio.create_task(work())

    await asyncio.sleep(1)

    print("4: cancelling") # 2
    task.cancel()

    await task 

    print("5: main finished") # 4

```text
create_task(work())
        ↓
Task is scheduled
        ↓
event loop runs work()
        ↓
"1: started"
        ↓
await asyncio.sleep(10)
        ↓
work() suspends
        ↓
main() continues
        ↓
after 1 second:
"4: cancelling task"
        ↓
task.cancel()
        ↓
cancellation is delivered into work()
        ↓
CancelledError occurs at the suspended await
        ↓
except asyncio.CancelledError runs
        ↓
"3: cancellation received"
        ↓
work() finishes
        ↓
await task completes
        ↓
"5: main finished"
```

In [ ]:
await main()

1: started
4: cancelling
3: cancellation error
5: main finished


In [ ]:
# Without try except
import asyncio 

async def work():
    print ("1: started") # 1


    await asyncio.sleep(10)
    print("2: finished sleeping")



async def main():
    task = asyncio.create_task(work())

    await asyncio.sleep(1)

    print("4: cancelling") # 2
    task.cancel() 

    await task 

    print("5: main finished")

In [ ]:
await main() # without the try except the cancellation raises CancelledError

1: started
4: cancelling


CancelledError: 

In [ ]:
async def work():
    try:
        print ("started")
        await asyncio.sleep(5)
        print ("finished")

    except asyncio.CancelledError:
        print ("cancelled")
        raise
        print("after raise") # Wont get printed as it is after raise

In [ ]:
await work()

started
finished


In [ ]:
task = asyncio.create_task(work())

started
cancelled


In [ ]:
task.cancel()

True

In [ ]:
async def work():
    try:
        print ("started")
        await asyncio.sleep(10)
        print ("finished")

    except asyncio.CancelledError:
        print ("cancelled")

async def main():
    task = asyncio.create_task(work())

    await asyncio.sleep(1)

    task.cancel()
    print("after cancel")

    await task

    print("main finished")

In [ ]:
await main()

started
after cancel
cancelled
main finished


#### Canceling parent 


In [ ]:
async def child():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("chid cancelled")
        raise

async def parent():
    task = asyncio.create_task(child())

    try:
        await task 
    except asyncio.CancelledError:
        print("parent cancelled")
        raise


In [ ]:
parent_task = asyncio.create_task(parent())

chid cancelled
parent cancelled


In [ ]:
parent_task.cancel()

True

In [ ]:
await parent_task

CancelledError: 

```text
parent receives cancellation request
        ↓
parent is currently awaiting child
        ↓
child is cancelled
        ↓
child receives CancelledError
        ↓
child finishes
        ↓
parent receives CancelledError
        ↓
parent finishes
```

In [ ]:
async def child():
    try:
        await asyncio.sleep(4)
        print("child finished")
    except asyncio.CancelledError:
        print("child cancelled")
        raise

async def parent():
    task = asyncio.create_task(child())
    try:
        await asyncio.sleep(1)
    except asyncio.CancelledError:
        print("parent cancelled")
        raise # means raise the same exception again after I have handled it.

In [ ]:
parent_task = asyncio.create_task(parent())


child finished


In [ ]:
parent_task.cancel()

False

In [ ]:
async def child():
    try:
        await asyncio.sleep(5)
        print("child finished")
    except asyncio.CancelledError:
        print("child cancelled")
        raise

async def parent():
    task = asyncio.create_task(child())

    await asyncio.sleep(1)

    task.cancel()

    print("parent continues")

    await task

In [ ]:
parent_task = asyncio.create_task(parent())

parent continues
child cancelled


#### gather() and cancel()

In [ ]:
async def A():
    try: 
        await asyncio.sleep(2)
        print("A finished")
        return ("A finished")
    except asyncio.CancelledError:
        print("A cancelled")
        raise

async def B():
    try: 
        await asyncio.sleep(4)
        print("B finished")
        return ("B finished")
    except asyncio.CancelledError:
        print("B cancelled")
        raise

In [ ]:
await asyncio.gather(A(),B())

A finished
B finished


['A finished', 'B finished']

In [ ]:
async def parent():
    await asyncio.gather(A(),B())

parent_task = asyncio.create_task(parent())

await asyncio.sleep(1)

parent_task.cancel()

True

A cancelled
B cancelled


In [ ]:
async def A():
    
    raise ValueError

async def B():
    try: 
        await asyncio.sleep(4)
        print("B finished")
        return ("B finished")
    except asyncio.CancelledError:
        print("B cancelled")
        raise

In [ ]:
results = await asyncio.gather(
    A(),
    B(),
    return_exceptions=True
) 

B finished


In [ ]:
print(results)

[ValueError(), 'B finished']


## Times out

#### `asyncio.wait_for()`

In [ ]:
# wait_for() and cancellation
async def slow_operation():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("operation cancelled")
        raise
    

In [ ]:
await asyncio.wait_for(slow_operation(), timeout=2) # Time out error not cancelled error

operation cancelled


TimeoutError: 

In [ ]:
# Suppressing CancelledError
async def slow_operation():
    try:
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("operation cancelled")
        # no raise
    

In [5]:
await asyncio.wait_for(slow_operation(), timeout=2) #  cancelled error bec there is no raise

operation cancelled


#### `asyncio.timeout()`

In [4]:
# Instead of await asyncio.wait_for (work(),timeout = 2)

async with asyncio.timeout(2):
    await slow_operation()

operation cancelled


In [6]:
async def work():
    try: 
        await asyncio.sleep(10)
    except asyncio.CancelledError:
        print("cleanup")
        raise

In [7]:
try:
    await asyncio.wait_for(work(), timeout=2)
except TimeoutError:
    print("timed out")

cleanup
timed out


```text
work()
  ↓
await asyncio.sleep(10)
  ↓
2 seconds pass
  ↓
wait_for() requests cancellation
  ↓
CancelledError delivered to work()
  ↓
except asyncio.CancelledError
  ↓
print("cleanup")
  ↓
raise
  ↓
cancellation propagates back to wait_for()
  ↓
wait_for() converts the timeout situation into TimeoutError
  ↓
except TimeoutError
  ↓
print("timed out")
```

In [31]:
async def get_user():
    await asyncio.sleep(1)
    return "User name"

async def get_preferences():
    await  asyncio.sleep(2)
    return "Good performance"

async def get_recommendations():
    await  asyncio.sleep(4)
    return "Recommendation is 1,2,3"


In [28]:
print(await get_user())
print(await get_preferences())
print(await get_recommendations())


User name
Good performance
Recommendation is 1,2,3


In [ ]:
# Sequential operations with a timeout
async with asyncio.timeout(5): # Everything inside the block has 5 second deadline.
    await get_user()
    await get_preferences()
    await get_recommendations()

TimeoutError: 

In [ ]:
# Concurrent operations with gather() and timeout.
async with asyncio.timeout(5): 
    result = await asyncio.gather (
        get_user(),
        get_preferences(),
        get_recommendations(),
    )

print (result) # All succeeded.

['User name', 'Good performance', 'Recommendation is 1,2,3']


In [ ]:
# Timeout while gather() is still running,
async with asyncio.timeout(4): 
    result = await asyncio.gather (
        get_user(),
        get_preferences(),
        get_recommendations(),
    )

print (result) # get_recommendations failed (5 sec and max timeout is 4).
                # 1 failed all fail.

TimeoutError: 